# Week 2: Feature Engineering & Data Preprocessing for Machine Learning

## Employee Attrition Analysis

**Data Science Internship Project**

This notebook continues the Week 1 analysis of the **IBM HR Analytics – Employee Attrition & Performance** dataset.

The goal of Week 2 is to transform the original HR dataset into:

- a **cleaned, human-readable dataset**;
- a **machine-learning-ready dataset**;
- a documented and reproducible preprocessing workflow for Week 3 model development.

> This notebook focuses on preprocessing and feature engineering. Final predictive model development is intentionally deferred to Week 3.


## 1. Business and Preprocessing Objectives

Employee attrition can create recruitment costs, productivity disruption, loss of institutional knowledge, and additional pressure on remaining employees.

The Week 1 exploratory analysis identified patterns involving overtime, business travel, job role, age, income, and organisational tenure.

### Week 2 Objectives

This notebook will:

1. Validate the quality and structure of the source dataset.
2. Remove non-informative and identifier variables.
3. Encode the target variable for classification.
4. Engineer meaningful HR-related features.
5. Separate the target from predictor variables.
6. Create a stratified train/test split.
7. Encode categorical variables.
8. Scale numerical variables.
9. Prevent preprocessing leakage by fitting transformations on the training set only.
10. Generate cleaned and machine-learning-ready CSV files.
11. Validate the final outputs before Week 3 modelling.


## 2. Import Libraries

In [1]:
import io
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Libraries imported successfully.")


Libraries imported successfully.


## 3. Load the Dataset

The cell below is designed for **Google Colab**.

When executed, it will prompt you to upload the original IBM HR Analytics CSV file.


In [2]:
try:
    from google.colab import files

    print("Upload: WA_Fn-UseC_-HR-Employee-Attrition.csv")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file was uploaded.")

    source_filename = next(iter(uploaded))
    df = pd.read_csv(io.BytesIO(uploaded[source_filename]))

except ImportError:
    DATA_PATH = Path("WA_Fn-UseC_-HR-Employee-Attrition.csv")

    if not DATA_PATH.exists():
        raise FileNotFoundError(
            "Dataset not found. Place WA_Fn-UseC_-HR-Employee-Attrition.csv "
            "in the same folder as this notebook."
        )

    source_filename = DATA_PATH.name
    df = pd.read_csv(DATA_PATH)

print(f"Loaded: {source_filename}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


Upload: WA_Fn-UseC_-HR-Employee-Attrition.csv


Saving WA_Fn-UseC_-HR-Employee-Attrition.csv to WA_Fn-UseC_-HR-Employee-Attrition.csv
Loaded: WA_Fn-UseC_-HR-Employee-Attrition.csv
Rows: 1,470
Columns: 35


In [3]:
df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


### Preserve the Raw Dataset

An untouched copy is retained so that preprocessing decisions can always be compared against the original data.


In [4]:
raw_df = df.copy()

print("Raw dataset copy preserved.")


Raw dataset copy preserved.


## 4. Initial Data Audit

In [5]:
print("Dataset shape:", raw_df.shape)
print("Total missing values:", raw_df.isna().sum().sum())
print("Duplicate rows:", raw_df.duplicated().sum())


Dataset shape: (1470, 35)
Total missing values: 0
Duplicate rows: 0


In [6]:
raw_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [7]:
audit_df = pd.DataFrame({
    "Column": raw_df.columns,
    "Data Type": [str(raw_df[col].dtype) for col in raw_df.columns],
    "Missing Values": [raw_df[col].isna().sum() for col in raw_df.columns],
    "Unique Values": [raw_df[col].nunique(dropna=False) for col in raw_df.columns]
})

audit_df.sort_values(
    ["Unique Values", "Missing Values"],
    ascending=[True, False]
)


,Column,Data Type,Missing Values,Unique Values
8,EmployeeCount,int64,0,1
21,Over18,object,0,1
26,StandardHours,int64,0,1
1,Attrition,object,0,2
11,Gender,object,0,2
22,OverTime,object,0,2
24,PerformanceRating,int64,0,2
2,BusinessTravel,object,0,3
4,Department,object,0,3
17,MaritalStatus,object,0,3


### Data Audit Interpretation

The audit confirms that the source dataset is structurally clean:

- **1,470 employee records** and **35 original variables**
- **0 missing values**
- **0 duplicate records**
- `EmployeeCount`, `Over18`, and `StandardHours` each contain only one unique value

Because constant variables cannot distinguish one employee from another, they do not provide predictive information and are removed before machine-learning preparation. `EmployeeNumber` is also excluded because it is an identifier rather than an employee characteristic.


## 5. Data Cleaning

The cleaning stage removes:

- exact duplicate records;
- constant variables that contain no variation;
- identifier columns that do not represent employee characteristics.

The target variable is retained and encoded separately.


In [8]:
# Identify constant columns and identifier fields that should not be used as predictors
cleaned_df = raw_df.drop_duplicates().copy()

constant_columns = [
    col for col in cleaned_df.columns
    if cleaned_df[col].nunique(dropna=False) == 1
]

identifier_columns = [
    col for col in ["EmployeeNumber"]
    if col in cleaned_df.columns
]

print("Constant columns:", constant_columns)
print("Identifier columns:", identifier_columns)


Constant columns: ['EmployeeCount', 'Over18', 'StandardHours']
Identifier columns: ['EmployeeNumber']


In [9]:
cleaned_df = cleaned_df.drop(
    columns=constant_columns + identifier_columns,
    errors="ignore"
)

print("Shape after removing non-informative columns:", cleaned_df.shape)


Shape after removing non-informative columns: (1470, 31)


## 6. Prepare the Target Variable

`Attrition` is the classification target.

It is converted to a binary numerical variable:

- `No` → `0`
- `Yes` → `1`


In [10]:
# Encode the binary target for future classification modelling
cleaned_df["AttritionFlag"] = (
    cleaned_df["Attrition"]
    .map({"No": 0, "Yes": 1})
    .astype(int)
)

cleaned_df[["Attrition", "AttritionFlag"]].value_counts().sort_index()


,,count
Attrition,AttritionFlag,
No,0,1233
Yes,1,237


### Target Distribution

The binary target preserves the original attrition distribution:

- **1,233 employees (83.88%)** remained
- **237 employees (16.12%)** left

This confirms that attrition is an **imbalanced classification problem**. The class imbalance should therefore be considered during Week 3 model selection and evaluation; accuracy alone will not be sufficient.


## 7. Feature Engineering

New variables are created from existing HR information to make workforce patterns easier for future models to learn.

### Engineered Features

- **AgeGroup** — groups employees into interpretable age ranges.
- **IncomeBand** — divides monthly income into quartile-based groups.
- **TenureGroup** — groups company tenure into business-friendly bands.
- **EarlyCareerFlag** — identifies employees with 5 or fewer total working years.
- **LongCommuteFlag** — identifies employees living 15 or more units from work.
- **YearsWithoutPromotion** — approximates the time between company tenure and the most recent promotion.
- **RoleTenureRatio** — proportion of company tenure spent in the current role.
- **ManagerTenureRatio** — proportion of company tenure spent with the current manager.
- **CompanyExperienceRatio** — proportion of total career experience spent at the current company.
- **OvertimeRiskFlag** — binary representation of overtime participation.

These features will later be evaluated during model development rather than assumed to be predictive.


In [11]:
# Create additional HR features using only information available in the source dataset
engineered_df = cleaned_df.copy()

engineered_df["AgeGroup"] = pd.cut(
    engineered_df["Age"],
    bins=[17, 25, 35, 45, 55, np.inf],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"]
)

engineered_df["IncomeBand"] = pd.qcut(
    engineered_df["MonthlyIncome"],
    q=4,
    labels=["Low", "Lower-Middle", "Upper-Middle", "High"],
    duplicates="drop"
)

engineered_df["TenureGroup"] = pd.cut(
    engineered_df["YearsAtCompany"],
    bins=[-1, 1, 3, 5, 10, np.inf],
    labels=["0-1 years", "2-3 years", "4-5 years", "6-10 years", "11+ years"]
)

engineered_df["EarlyCareerFlag"] = (
    engineered_df["TotalWorkingYears"] <= 5
).astype(int)

engineered_df["LongCommuteFlag"] = (
    engineered_df["DistanceFromHome"] >= 15
).astype(int)

engineered_df["YearsWithoutPromotion"] = (
    engineered_df["YearsAtCompany"]
    - engineered_df["YearsSinceLastPromotion"]
).clip(lower=0)

engineered_df["RoleTenureRatio"] = np.where(
    engineered_df["YearsAtCompany"] > 0,
    engineered_df["YearsInCurrentRole"] / engineered_df["YearsAtCompany"],
    0
)

engineered_df["ManagerTenureRatio"] = np.where(
    engineered_df["YearsAtCompany"] > 0,
    engineered_df["YearsWithCurrManager"] / engineered_df["YearsAtCompany"],
    0
)

engineered_df["CompanyExperienceRatio"] = np.where(
    engineered_df["TotalWorkingYears"] > 0,
    engineered_df["YearsAtCompany"] / engineered_df["TotalWorkingYears"],
    0
)

engineered_df["OvertimeRiskFlag"] = (
    engineered_df["OverTime"] == "Yes"
).astype(int)

print("Feature engineering completed.")
print("Current shape:", engineered_df.shape)


Feature engineering completed.
Current shape: (1470, 42)


In [12]:
engineered_features = [
    "AgeGroup",
    "IncomeBand",
    "TenureGroup",
    "EarlyCareerFlag",
    "LongCommuteFlag",
    "YearsWithoutPromotion",
    "RoleTenureRatio",
    "ManagerTenureRatio",
    "CompanyExperienceRatio",
    "OvertimeRiskFlag"
]

engineered_df[engineered_features].head()


,AgeGroup,IncomeBand,TenureGroup,EarlyCareerFlag,LongCommuteFlag,YearsWithoutPromotion,RoleTenureRatio,ManagerTenureRatio,CompanyExperienceRatio,OvertimeRiskFlag
0,36-45,Upper-Middle,6-10 years,0,0,6,0.67,0.83,0.75,1
1,46-55,Upper-Middle,6-10 years,0,0,9,0.70,0.70,1.00,0
2,36-45,Low,0-1 years,0,0,0,0.00,0.00,0.00,1
3,26-35,Low,6-10 years,0,0,5,0.88,0.00,1.00,1
4,26-35,Lower-Middle,2-3 years,0,0,0,1.00,1.00,0.33,0


### Feature Engineering Rationale

Ten additional features were created to capture employee characteristics in forms that may be easier for future models to use.

The engineered features deliberately retain the original variables as well. This allows Week 3 model development to test whether the derived features add useful predictive information rather than assuming that they do.

Some engineered features are correlated with their source variables—for example, `AgeGroup` with `Age`, and `OvertimeRiskFlag` with `OverTime`. Their usefulness should therefore be assessed during feature selection and model evaluation.


## 8. Validate the Cleaned Dataset

In [13]:
cleaned_validation = pd.DataFrame({
    "Check": [
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows",
        "Target Missing Values",
        "Target Classes"
    ],
    "Result": [
        len(engineered_df),
        engineered_df.shape[1],
        int(engineered_df.isna().sum().sum()),
        int(engineered_df.duplicated().sum()),
        int(engineered_df["AttritionFlag"].isna().sum()),
        int(engineered_df["AttritionFlag"].nunique())
    ]
})

cleaned_validation


,Check,Result
0,Rows,1470
1,Columns,42
2,Missing Values,0
3,Duplicate Rows,0
4,Target Missing Values,0
5,Target Classes,2


### Cleaned Dataset Validation

After cleaning and feature engineering, the dataset contains:

- **1,470 rows**
- **42 columns**
- **0 missing values**
- **0 duplicate rows**
- **2 valid target classes**

The row count remains unchanged, confirming that no employee records were unintentionally lost during preprocessing.


## 9. Separate Features and Target

The original text target (`Attrition`) and binary target (`AttritionFlag`) are excluded from the predictor matrix.

The binary target is stored separately as `y`.


In [14]:
X = engineered_df.drop(
    columns=["Attrition", "AttritionFlag"],
    errors="ignore"
)

y = engineered_df["AttritionFlag"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
display(y.value_counts().to_frame("Employees"))
display((y.value_counts(normalize=True) * 100).round(2).to_frame("Percentage"))


Feature matrix shape: (1470, 40)
Target shape: (1470,)

Target distribution:


,Employees
AttritionFlag,
0,1233
1,237


,Percentage
AttritionFlag,
0,83.88
1,16.12


## 10. Train/Test Split

A stratified split is used so that the proportion of attrition cases remains similar in the training and testing datasets.

- Training set: **80%**
- Test set: **20%**
- Random state: **42**

The split is performed **before fitting the encoder and scaler** to reduce preprocessing leakage.


In [15]:
# Split before fitting preprocessing transformations to reduce data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows: {len(X_test):,}")
print(f"Training attrition rate: {y_train.mean() * 100:.2f}%")
print(f"Testing attrition rate: {y_test.mean() * 100:.2f}%")


Training rows: 1,176
Testing rows: 294
Training attrition rate: 16.16%
Testing attrition rate: 15.99%


### Train/Test Split Validation

The stratified split produced:

- **1,176 training records (80%)**
- **294 testing records (20%)**
- Training attrition rate: **16.16%**
- Testing attrition rate: **15.99%**

The nearly identical attrition proportions show that stratification successfully preserved the target distribution across both subsets.


## 11. Identify Numerical and Categorical Features

In [16]:
numeric_columns = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_columns = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print(f"Numerical features: {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")

print("\nNumerical columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)


Numerical features: 30
Categorical features: 10

Numerical columns:
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'EarlyCareerFlag', 'LongCommuteFlag', 'YearsWithoutPromotion', 'RoleTenureRatio', 'ManagerTenureRatio', 'CompanyExperienceRatio', 'OvertimeRiskFlag']

Categorical columns:
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'AgeGroup', 'IncomeBand', 'TenureGroup']


### Feature-Type Summary

Before transformation, the predictor set contains:

- **30 numerical features**
- **10 categorical features**

Numerical and categorical variables are processed separately because they require different transformations. Numerical variables are standardised, while categorical variables are one-hot encoded.


## 12. Encode and Scale Features

### Numerical Variables
`StandardScaler` standardises numerical features using the mean and standard deviation learned from the training set.

### Categorical Variables
`OneHotEncoder` converts nominal categorical variables into binary indicator columns.

`handle_unknown="ignore"` ensures unseen categories in the test set do not cause transformation errors.

The preprocessing transformer is **fit only on the training data** and then applied to both training and testing data.


In [17]:
# Fit scaling and encoding on the training set only, then transform the test set
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_columns
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        )
    ],
    remainder="drop"
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Preprocessing completed.")
print("Final transformed feature count:", len(feature_names))


Preprocessing completed.
Final transformed feature count: 72


### Leakage-Control Decision

The preprocessing transformer is fitted using **only the training data**.

This is important because fitting the scaler or encoder on the full dataset would allow information from the test set to influence preprocessing. Keeping the test set unseen until transformation provides a more realistic foundation for Week 3 model evaluation.


## 13. Build the Machine-Learning-Ready Dataset

In [18]:
# Rebuild the transformed arrays as labelled DataFrames for transparency and export
X_train_ready = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

X_test_ready = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

train_ready = X_train_ready.copy()
train_ready["Attrition"] = y_train
train_ready["DatasetSplit"] = "Train"

test_ready = X_test_ready.copy()
test_ready["Attrition"] = y_test
test_ready["DatasetSplit"] = "Test"

ml_ready_df = (
    pd.concat([train_ready, test_ready], axis=0)
    .sort_index()
    .reset_index(drop=True)
)

print("ML-ready dataset shape:", ml_ready_df.shape)
ml_ready_df.head()


ML-ready dataset shape: (1470, 74)


,numeric__Age,numeric__DailyRate,numeric__DistanceFromHome,numeric__Education,numeric__EnvironmentSatisfaction,numeric__HourlyRate,numeric__JobInvolvement,numeric__JobLevel,numeric__JobSatisfaction,numeric__MonthlyIncome,numeric__MonthlyRate,numeric__NumCompaniesWorked,numeric__PercentSalaryHike,numeric__PerformanceRating,numeric__RelationshipSatisfaction,numeric__StockOptionLevel,numeric__TotalWorkingYears,numeric__TrainingTimesLastYear,numeric__WorkLifeBalance,numeric__YearsAtCompany,numeric__YearsInCurrentRole,numeric__YearsSinceLastPromotion,numeric__YearsWithCurrManager,numeric__EarlyCareerFlag,numeric__LongCommuteFlag,numeric__YearsWithoutPromotion,numeric__RoleTenureRatio,numeric__ManagerTenureRatio,numeric__CompanyExperienceRatio,numeric__OvertimeRiskFlag,categorical__BusinessTravel_Non-Travel,categorical__BusinessTravel_Travel_Frequently,categorical__BusinessTravel_Travel_Rarely,categorical__Department_Human Resources,categorical__Department_Research & Development,categorical__Department_Sales,categorical__EducationField_Human Resources,categorical__EducationField_Life Sciences,categorical__EducationField_Marketing,categorical__EducationField_Medical,categorical__EducationField_Other,categorical__EducationField_Technical Degree,categorical__Gender_Female,categorical__Gender_Male,categorical__JobRole_Healthcare Representative,categorical__JobRole_Human Resources,categorical__JobRole_Laboratory Technician,categorical__JobRole_Manager,categorical__JobRole_Manufacturing Director,categorical__JobRole_Research Director,categorical__JobRole_Research Scientist,categorical__JobRole_Sales Executive,categorical__JobRole_Sales Representative,categorical__MaritalStatus_Divorced,categorical__MaritalStatus_Married,categorical__MaritalStatus_Single,categorical__OverTime_No,categorical__OverTime_Yes,categorical__AgeGroup_18-25,categorical__AgeGroup_26-35,categorical__AgeGroup_36-45,categorical__AgeGroup_46-55,categorical__AgeGroup_56+,categorical__IncomeBand_High,categorical__IncomeBand_Low,categorical__IncomeBand_Lower-Middle,categorical__IncomeBand_Upper-Middle,categorical__TenureGroup_0-1 years,categorical__TenureGroup_11+ years,categorical__TenureGroup_2-3 years,categorical__TenureGroup_4-5 years,categorical__TenureGroup_6-10 years,Attrition,DatasetSplit
0,0.44,0.74,-1.02,-0.88,-0.66,1.40,0.37,-0.07,1.15,-0.12,0.71,2.14,-1.15,-0.43,-1.60,-0.94,-0.43,-2.20,-2.45,-0.17,-0.06,-0.68,0.23,-0.52,-0.58,0.23,0.26,0.80,0.21,1.57,0.00,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,1,Train
1,1.31,-1.31,-0.17,-1.86,0.26,-0.22,-1.05,-0.07,-0.65,-0.30,1.46,-0.68,2.11,2.31,1.16,0.25,-0.18,0.19,0.34,0.48,0.78,-0.37,0.79,-0.52,-0.58,0.85,0.36,0.40,0.98,-0.64,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0,Train
2,0.00,1.42,-0.90,-0.88,1.18,1.30,-1.05,-0.99,0.25,-0.96,-1.67,1.33,-0.07,-0.43,-0.68,-0.94,-0.56,0.19,0.34,-1.16,-1.19,-0.68,-1.18,-0.52,-0.58,-1.00,-1.75,-1.71,-2.09,1.57,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1,Test
3,-0.44,1.47,-0.78,1.06,1.18,-0.47,0.37,-0.99,0.25,-0.78,1.22,-0.68,-1.15,-0.43,0.24,-0.94,-0.43,0.19,0.34,0.16,0.78,0.25,-1.18,-0.52,-0.58,0.03,0.89,-1.71,0.98,1.57,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0,Test
4,-1.09,-0.53,-0.90,-1.86,-1.58,-1.25,0.37,-0.99,-0.65,-0.66,0.31,2.54,-0.88,-0.43,1.16,0.25,-0.69,0.19,0.34,-0.83,-0.63,-0.06,-0.62,-0.52,-0.58,-1.00,1.27,1.30,-1.0

### ML-Ready Dataset Result

After scaling and one-hot encoding:

- the original predictor set is transformed into **72 machine-learning features**;
- the final exported table contains **74 columns**, including the target and dataset-split indicator;
- all model-input features are numerical.

The `DatasetSplit` field is retained only to identify training and testing observations in the exported file. It should **not** be used as a predictor during model training.


## 14. Validate the Machine-Learning-Ready Dataset

In [19]:
ml_feature_columns = [
    col for col in ml_ready_df.columns
    if col not in ["Attrition", "DatasetSplit"]
]

non_numeric_features = (
    ml_ready_df[ml_feature_columns]
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

ml_validation = pd.DataFrame({
    "Check": [
        "Rows",
        "Columns",
        "Missing Values",
        "Non-Numeric Feature Columns",
        "Target Classes",
        "Training Rows",
        "Testing Rows"
    ],
    "Result": [
        len(ml_ready_df),
        ml_ready_df.shape[1],
        int(ml_ready_df.isna().sum().sum()),
        len(non_numeric_features),
        int(ml_ready_df["Attrition"].nunique()),
        int((ml_ready_df["DatasetSplit"] == "Train").sum()),
        int((ml_ready_df["DatasetSplit"] == "Test").sum())
    ]
})

ml_validation


,Check,Result
0,Rows,1470
1,Columns,74
2,Missing Values,0
3,Non-Numeric Feature Columns,0
4,Target Classes,2
5,Training Rows,1176
6,Testing Rows,294


### Final Validation

The machine-learning-ready dataset passed the main preprocessing checks:

- **1,470 rows retained**
- **0 missing values**
- **0 non-numeric predictor columns**
- **2 target classes**
- **1,176 training rows**
- **294 testing rows**

These checks confirm that the generated dataset is suitable for the next modelling stage.


## 15. Preprocessing Summary

In [20]:
preprocessing_summary = pd.DataFrame({
    "Item": [
        "Source rows",
        "Source columns",
        "Constant columns removed",
        "Identifier columns removed",
        "Engineered features created",
        "Numerical features",
        "Categorical features",
        "Final ML features",
        "Training rows",
        "Testing rows",
        "Training attrition rate (%)",
        "Testing attrition rate (%)"
    ],
    "Value": [
        raw_df.shape[0],
        raw_df.shape[1],
        ", ".join(constant_columns),
        ", ".join(identifier_columns),
        len(engineered_features),
        len(numeric_columns),
        len(categorical_columns),
        len(feature_names),
        len(X_train),
        len(X_test),
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

preprocessing_summary


,Item,Value
0,Source rows,1470
1,Source columns,35
2,Constant columns removed,"EmployeeCount, Over18, StandardHours"
3,Identifier columns removed,EmployeeNumber
4,Engineered features created,10
5,Numerical features,30
6,Categorical features,10
7,Final ML features,72
8,Training rows,1176
9,Testing rows,294


### Preprocessing Outcome

The Week 2 workflow reduced the original dataset from **35 raw variables** to a cleaned analytical structure, created **10 engineered features**, and produced **72 transformed ML features** after encoding and scaling.

The final train and test sets retain nearly identical attrition rates, supporting reliable model comparison in Week 3.


## 16. Export Final Deliverables

The notebook generates:

1. `cleaned_employee_attrition.csv`  
   Human-readable cleaned dataset with engineered features.

2. `employee_attrition_ml_ready.csv`  
   Encoded and scaled dataset ready for Week 3 modelling.

3. `week2_data_audit.csv`  
   Column-level data quality audit.

4. `week2_preprocessing_summary.csv`  
   Summary of the preprocessing workflow.


In [21]:
engineered_df.to_csv(
    "cleaned_employee_attrition.csv",
    index=False
)

ml_ready_df.to_csv(
    "employee_attrition_ml_ready.csv",
    index=False
)

audit_df.to_csv(
    "week2_data_audit.csv",
    index=False
)

preprocessing_summary.to_csv(
    "week2_preprocessing_summary.csv",
    index=False
)

output_files = [
    "cleaned_employee_attrition.csv",
    "employee_attrition_ml_ready.csv",
    "week2_data_audit.csv",
    "week2_preprocessing_summary.csv"
]

print("Files created successfully:")
for file in output_files:
    print("-", file)


Files created successfully:
- cleaned_employee_attrition.csv
- employee_attrition_ml_ready.csv
- week2_data_audit.csv
- week2_preprocessing_summary.csv


### Download the Outputs in Google Colab

Run the next cell after all previous cells have completed successfully.


In [22]:
try:
    from google.colab import files

    for file in output_files:
        files.download(file)

except ImportError:
    print("Automatic download is available when running in Google Colab.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>